In [18]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [19]:
class IndoorScenesDataset(Dataset):
    def __init__(self, data, label_to_idx, transform=None):
        self.data = data
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.label_to_idx[label]


In [20]:
def patchify(imgs, patch_size=16):
    B, C, H, W = imgs.shape
    h = H // patch_size
    w = W // patch_size
    patches = imgs.reshape(B, C, h, patch_size, w, patch_size)
    patches = patches.permute(0, 2, 4, 3, 5, 1).reshape(B, h * w, patch_size * patch_size * C)
    return patches


In [21]:
def random_masking(x, mask_ratio):
    N, L, D = x.shape
    len_keep = int(L * (1 - mask_ratio))
    noise = torch.rand(N, L, device=x.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)
    ids_keep = ids_shuffle[:, :len_keep]
    x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
    mask = torch.ones([N, L], device=x.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)
    return x_masked, mask, ids_restore


In [22]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class SimpleViTEncoder(nn.Module):
    def __init__(self, embed_dim=768, depth=4, num_heads=8):
        super().__init__()
        self.patch_embed = PatchEmbed(embed_dim=embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, self.patch_embed.num_patches, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.embed_dim = embed_dim

    def forward_encoder(self, x):
        x = x + self.pos_embed[:, :x.size(1), :]
        return self.encoder(x)

class MAEViT(nn.Module):
    def __init__(self, encoder, decoder_dim=512, mask_ratio=0.75):
        super().__init__()
        self.encoder = encoder
        self.decoder = nn.Sequential(
            nn.Linear(encoder.embed_dim, decoder_dim),
            nn.ReLU(),
            nn.Linear(decoder_dim, 16 * 16 * 3)
        )
        self.mask_ratio = mask_ratio

    def forward(self, imgs):
        patches = self.encoder.patch_embed(imgs)
        x_masked, mask, ids_restore = random_masking(patches, self.mask_ratio)
        encoded = self.encoder.forward_encoder(x_masked)
        decoded = self.decoder(encoded)
        return decoded, mask


In [23]:
class ViTClassifier(nn.Module):
    def __init__(self, encoder, num_classes=67):
        super().__init__()
        self.encoder = encoder
        self.cls_head = nn.Linear(encoder.embed_dim, num_classes)

    def forward(self, x):
        x = self.encoder.patch_embed(x)
        x = self.encoder.forward_encoder(x)
        cls_token = x.mean(dim=1)
        return self.cls_head(cls_token)


In [24]:
# def train_mae(model, dataloader, optimizer, device, epochs=20):
#     model.train()
#     loss_fn = nn.MSELoss()
#     scaler = torch.cuda.amp.GradScaler()
#     for epoch in range(epochs):
#         total_loss = 0
#         for imgs, _ in dataloader:
#             imgs = imgs.to(device)
#             with torch.cuda.amp.autocast():
#                 preds, mask = model(imgs)
#                 gt = patchify(imgs)
#                 loss = loss_fn(preds, gt)

#             optimizer.zero_grad()
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()
#             total_loss += loss.item()

#         print(f"[Epoch {epoch}] MAE Loss: {total_loss / len(dataloader):.4f}")
#         torch.save(model.encoder.state_dict(), "mae_vit_encoder_best.pth")
def train_mae(model, dataloader, optimizer, device, epochs=20):
    model.train()
    loss_fn = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(epochs):
        total_loss = 0
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            with torch.cuda.amp.autocast():
                preds, mask = model(imgs)
                gt = patchify(imgs)

                # Keep only masked patches (preds are only for masked ones)
                B, L, D = gt.shape
                num_visible = preds.shape[1]
                ids_restore = torch.argsort(mask, dim=1)  # sort by unmask first
                ids_masked = ids_restore[:, -num_visible:]

                gt_masked = torch.gather(gt, 1, ids_masked.unsqueeze(-1).expand(-1, -1, D))

                loss = loss_fn(preds, gt_masked)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        print(f"[Epoch {epoch}] MAE Loss: {total_loss / len(dataloader):.4f}")
        torch.save(model.encoder.state_dict(), "mae_vit_encoder_best.pth")


In [25]:
def train_classifier(model, train_loader, val_loader, device, epochs=20):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-4, steps_per_epoch=len(train_loader), epochs=epochs
    )
    scaler = torch.cuda.amp.GradScaler()
    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()

        acc = evaluate(model, val_loader, device)
        print(f"Epoch {epoch}: Val Accuracy = {acc:.2f}%")
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), "vit_finetuned_best.pth")


In [26]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

def tta_predict(model, img, device, transforms_list):
    model.eval()
    logits = []
    for t in transforms_list:
        aug_img = t(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(aug_img)
            logits.append(out)
    avg_logits = torch.mean(torch.stack(logits), dim=0)
    return torch.argmax(avg_logits, dim=1)


In [27]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

def tta_predict(model, img, device, transforms_list):
    model.eval()
    logits = []
    for t in transforms_list:
        aug_img = t(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(aug_img)
            logits.append(out)
    avg_logits = torch.mean(torch.stack(logits), dim=0)
    return torch.argmax(avg_logits, dim=1)


In [28]:
from torchvision.datasets.folder import default_loader
import kagglehub
# Setup root directory
itsahmad_indoor_scenes_cvpr_2019_path = kagglehub.dataset_download('itsahmad/indoor-scenes-cvpr-2019')
print('Data source import complete.')
basePath = itsahmad_indoor_scenes_cvpr_2019_path

root_dir = basePath + "/indoorCVPR_09/Images"

# Read all image paths and labels
all_data = []
for label in sorted(os.listdir(root_dir)):
    label_dir = os.path.join(root_dir, label)
    for file in os.listdir(label_dir):
        if file.endswith(('.jpg', '.png')):
            all_data.append((os.path.join(label_dir, file), label))

# Label mapping
label_names = sorted(set([label for _, label in all_data]))
label_to_idx = {label: idx for idx, label in enumerate(label_names)}

# Stratified split
train_data, temp_data = train_test_split(
    all_data, test_size=0.2, stratify=[label for _, label in all_data], random_state=42
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, stratify=[label for _, label in temp_data], random_state=42
)

# Transforms
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

mae_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.2),
    transforms.RandomAffine(degrees=15, shear=10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


Data source import complete.


In [29]:
batch_size = 64

mae_dataset = IndoorScenesDataset(train_data + val_data + test_data, label_to_idx, transform=mae_transform)
# mae_loader = DataLoader(mae_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

train_dataset = IndoorScenesDataset(train_data, label_to_idx, transform=train_transform)
val_dataset = IndoorScenesDataset(val_data, label_to_idx, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

mae_loader = DataLoader(mae_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


In [30]:
# Step 1: MAE Pretraining
encoder = SimpleViTEncoder(embed_dim=768, depth=4, num_heads=8).to(device)
mae_model = MAEViT(encoder=encoder).to(device)

mae_optimizer = optim.Adam(mae_model.parameters(), lr=1e-4)
print("🔁 Starting MAE Pretraining...")
train_mae(mae_model, mae_loader, mae_optimizer, device, epochs=20)

# Step 2: Fine-tuning for Classification
print("🎯 Starting Supervised Fine-Tuning...")
vit_model = ViTClassifier(encoder).to(device)
train_classifier(vit_model, train_loader, val_loader, device, epochs=20)


🔁 Starting MAE Pretraining...


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/var/folders/9_/cv27drrj3759y9yrmhdcbnv40000gn/T/ipykernel_28271/2510240347.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/opt/anaconda3/lib/python3.12/site-packages/torch/amp/grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
/var/folders/9_/cv27drrj3759y9yrmhdcbnv40000gn/T/ipykernel_28271/2510240347.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/opt/anaconda3/lib/python3.12/site-packages/to

[Epoch 0] MAE Loss: 1.2127
[Epoch 1] MAE Loss: 1.1925
[Epoch 2] MAE Loss: 1.1903
[Epoch 3] MAE Loss: 1.1882
[Epoch 4] MAE Loss: 1.1871
[Epoch 5] MAE Loss: 1.1862


KeyboardInterrupt: 